# Reproducible Design & Analysis: Common-Source Amplifier (VN2222LL)

**EL2205 Major Assignment** **Electrical Engineering Study Program - Bandung Institute of Technology (ITB)**

**Author:** Benedictus Kenneth Setiadi (13224003)  
**Date:** April 2026

-----

## 1\. Executive Summary

This notebook presents reproducible, end-to-end documentation for the design of a Common-Source voltage amplifier using a discrete VN2222LL MOSFET. Calculations are performed using high-precision representation (64-bit floating point) to prevent the accumulation of rounding errors during design iterations. This approach ensures that the target specifications can be met with a theoretical error approaching 0%.

## 2\. Target Specifications

Based on the assignment specification document, the parameters that must be met are:

  * **Topology:** Common-Source Amplifier
  * **Voltage Gain ($A_v$):** 10 V/V (20 dB)
  * **Input Resistance ($R_{in}$):** 27 kΩ
  * **Output Resistance ($R_{out}$):** 1.5 kΩ
  * **Supply Voltage ($V_{DD}$):** 12.0 V

## 3\. Topology & Circuit Schematic

This design utilizes a Common-Source architecture with the addition of Partial Source Degeneration to stabilize the DC operating point (auto-correct) while maintaining the linearity of the AC gain against transconductance ($g_m$) fluctuations.

\<p align="center"\>
\<img src="schematic.jpg" alt="Circuit Schematic" width="600"/\>
<br>
\<i\>Figure 1: Final schematic of the VN2222LL amplifier circuit. (Replace 'schematic.jpg' with your image file name)\</i\>
\</p\>

## 4\. Design Flow & Methodology

The design is executed using a reverse-flow approach, starting from establishing the absolute output resistance, designing the DC swing headroom, extracting silicon parasitic parameters, and finally establishing the input resistance.

\<p align="center"\>
\<img src="flowchart\_path.png" alt="Design Flowchart" width="400"/\>
<br>
\<i\>Figure 2: Flowchart of the design sequence. (Replace 'flowchart\_path.png' with your image file name)\</i\>
\</p\>

## 5\. Computational Engine

The code block below sequentially executes all mathematical equations. Silicon extraction parameters ($V_{th}$, $K_n$, $\lambda$) are isolated in the top block so the computation can be instantly reproduced when the laboratory testing data is updated.

-----

In [ ]:
%pip install PyLTSpice matplotlib numpy pandas scipy

In [ ]:
import numpy as np
import pandas as pd
from PyLTSpice import LTSpiceBatch

In [ ]:
# ==============================================================================
# BLOK 1: PARAMETER KONSTAN & TARGET SPESIFIKASI (UBAH DI SINI)
# ==============================================================================
# Parameter MOSFET (Contoh: hasil rata-rata tes lab dari 10 orang)
V_TH = np.float64(1.7115)           # Threshold Voltage (V)
K_N  = np.float64(23.45e-3)       # Parameter Konduksi (A/V^2) 1/2 * μ_n * C_ox * (W/L)
LAMBDA = np.float64(0.009)      # Parameter Modulasi Panjang Kanal (V^-1)

# Spesifikasi Target & Asumsi Desain
V_DD = np.float64(12.0)          # Tegangan Supply (V)
TARGET_AV = np.float64(10.0)     # Magnitude Gain |Av| (V/V) setara 20dB
TARGET_ROUT = np.float64(1500.0) # Target Rout (Ohm)
TARGET_RIN = np.float64(27000.0) # Target Rin (Ohm)

# Asumsi awal untuk ruang swing
TARGET_VD = np.float64(7.2)      
TARGET_VS = np.float64(1.5)


# ==============================================================================
# BLOK 2: FUNGSI HELPER (SOLVER & E24 MATCHER)
# ==============================================================================
def get_closest_e24(val):
    """Mencari nilai resistor standar E24 terdekat dengan presisi tinggi."""
    # Tambahkan 10.0 agar angka di atas 9.1 bisa membulat ke atas (ke orde berikutnya)
    e24_series = np.array([1.0, 1.1, 1.2, 1.3, 1.5, 1.6, 1.8, 2.0, 2.2, 2.4, 
                           2.7, 3.0, 3.3, 3.6, 3.9, 4.3, 4.7, 5.1, 5.6, 6.2, 
                           6.8, 7.5, 8.2, 9.1, 10.0], dtype=np.float64)
    if val <= 0: return np.float64(0)

    power = np.floor(np.log10(val))
    base_val = val / (np.float64(10)**power)
    
    idx = (np.abs(e24_series - base_val)).argmin()
    res = e24_series[idx] * (np.float64(10)**power)
    
    return np.float64(res)

def solve_bias_id(v_g, r_s_tot):
    """
    Menyelesaikan persamaan kuadrat I_D akibat feedback negatif DC dari R_S.
    Mengembalikan akar I_D yang secara fisika valid (V_GS > V_TH).
    """
    a = K_N * (r_s_tot**2)
    b = -(np.float64(2.0) * K_N * r_s_tot * (v_g - V_TH) + np.float64(1.0))
    c = K_N * ((v_g - V_TH)**2)
    
    id_1 = (-b + np.sqrt(b**2 - np.float64(4.0)*a*c)) / (np.float64(2.0)*a)
    id_2 = (-b - np.sqrt(b**2 - np.float64(4.0)*a*c)) / (np.float64(2.0)*a)
    
    vgs_1 = v_g - (id_1 * r_s_tot)
    return id_1 if vgs_1 > V_TH else id_2


# ==============================================================================
# BLOK 3: PERHITUNGAN JALUR 1 (MATEMATIKA EKSAK)
# ==============================================================================
# 1. Komponen & Bias Awalk
RD_ex = np.float64(1500.0) # Sengaja diambil >1500 agar paralelnya bisa turun
ID_ex = (V_DD - TARGET_VD) / RD_ex
RStot_ex = TARGET_VS / ID_ex
VS_ex = ID_ex * RStot_ex
VD_ex = V_DD - (ID_ex * RD_ex)

# 2. Parameter Sinyal Kecil Eksak
VGS_ex = np.sqrt(ID_ex / K_N) + V_TH
gm_ex = (np.float64(2.0) * ID_ex) / (VGS_ex - V_TH)
ro_ex = np.float64(1.0) / (LAMBDA * ID_ex)

# 3. Resistor Jaringan AC & DC Eksak
RS1_ex = ((gm_ex * ro_ex * RD_ex) - (TARGET_AV * (RD_ex + ro_ex))) / (TARGET_AV * (np.float64(1.0) + (gm_ex * ro_ex)))
RS2_ex = RStot_ex - RS1_ex

# 4. Resistor Input Eksak
VG_ex = VGS_ex + TARGET_VS
R1_ex = V_DD * (TARGET_RIN / VG_ex)
R2_ex = (R1_ex * TARGET_RIN) / (R1_ex - TARGET_RIN)

# 5. Output Eksak (Double Check)
Rin_ex = (R1_ex * R2_ex) / (R1_ex + R2_ex)
Rout_drain_ex = ro_ex + RS1_ex + (gm_ex * ro_ex * RS1_ex)
Rout_ex = (RD_ex * Rout_drain_ex) / (RD_ex + Rout_drain_ex)
Av_ex = (gm_ex * ro_ex * RD_ex) / (RD_ex + RS1_ex + ro_ex + (gm_ex * ro_ex * RS1_ex))


# ==============================================================================
# BLOK 4: PERHITUNGAN JALUR 2 (REALITA FISIKA - E24)
# ==============================================================================
# 1. Konversi ke standar komponen pasaran
RD_e24  = get_closest_e24(RD_ex)
RS1_e24 = get_closest_e24(RS1_ex)
RS2_e24 = get_closest_e24(RS2_ex)
R1_e24  = get_closest_e24(R1_ex)
R2_e24  = get_closest_e24(R2_ex)

# 2. Pergeseran Bias DC akibat E24
VG_e24 = V_DD * (R2_e24 / (R1_e24 + R2_e24))
RStot_e24 = RS1_e24 + RS2_e24
ID_e24 = solve_bias_id(VG_e24, RStot_e24)

VS_e24 = ID_e24 * RStot_e24
VD_e24 = V_DD - (ID_e24 * RD_e24)
VGS_e24 = VG_e24 - VS_e24

# 3. Pergeseran Parameter Sinyal Kecil akibat ID baru
gm_e24 = (np.float64(2.0) * ID_e24) / (VGS_e24 - V_TH)
ro_e24 = np.float64(1.0) / (LAMBDA * ID_e24)

# 4. Finalisasi Output AC dengan model E24
Rin_e24 = (R1_e24 * R2_e24) / (R1_e24 + R2_e24)
Rout_drain_e24 = ro_e24 + RS1_e24 + (gm_e24 * ro_e24 * RS1_e24)
Rout_e24 = (RD_e24 * Rout_drain_e24) / (RD_e24 + Rout_drain_e24)
Av_e24 = (gm_e24 * ro_e24 * RD_e24) / (RD_e24 + RS1_e24 + ro_e24 + (gm_e24 * ro_e24 * RS1_e24))

# ==============================================================================
# BLOK 5: UJI KETAHANAN AUTO-CORRECT (MENGGUNAKAN BIAS E24)
# ==============================================================================
ID_50mV  = solve_bias_id(VG_e24 + np.float64(0.05), RStot_e24)
ID_200mV = solve_bias_id(VG_e24 + np.float64(0.20), RStot_e24)
ID_500mV = solve_bias_id(VG_e24 + np.float64(0.50), RStot_e24)

# ==============================================================================
# BLOK 6: KALKULASI GALAT & RENDER TABEL (FULL REPRODUCIBLE)
# ==============================================================================
# Helper format untuk mengatasi nilai kosong di target
def calc_err(target, actual):
    return np.abs(target - actual) / target * 100

data = [
    # KOMPONEN RESISTOR
    ("Komponen", "R_D (Ohm)", "-", RD_ex, RD_e24, "-", "-"),
    ("Komponen", "R_S1 (Ohm)", "-", RS1_ex, RS1_e24, "-", "-"),
    ("Komponen", "R_S2 (Ohm)", "-", RS2_ex, RS2_e24, "-", "-"),
    ("Komponen", "R_1 (Ohm)", "-", R1_ex, R1_e24, "-", "-"),
    ("Komponen", "R_2 (Ohm)", "-", R2_ex, R2_e24, "-", "-"),
    
    # BIAS DC
    ("Bias DC", "V_G (V)", "-", VG_ex, VG_e24, "-", "-"),
    ("Bias DC", "V_S (V)", TARGET_VS, VS_ex, VS_e24, "-", "-"),
    ("Bias DC", "V_D (V)", TARGET_VD, VD_ex, VD_e24, "-", "-"),
    ("Bias DC", "V_GS (V)", "-", VGS_ex, VGS_e24, "-", "-"),
    ("Bias DC", "I_D (mA)", "-", ID_ex * 1000, ID_e24 * 1000, "-", "-"),
    
    # PARAMETER SMALL SIGNAL
    ("Sinyal Kecil", "g_m (A/V)", "-", gm_ex, gm_e24, "-", "-"),
    ("Sinyal Kecil", "r_o (Ohm)", "-", ro_ex, ro_e24, "-", "-"),
    
    # PERFORMA AC (YANG DINILAI ASISTEN)
    ("Performa AC", "R_in (Ohm)", TARGET_RIN, Rin_ex, Rin_e24, calc_err(TARGET_RIN, Rin_ex), calc_err(TARGET_RIN, Rin_e24)),
    ("Performa AC", "R_out (Ohm)", TARGET_ROUT, Rout_ex, Rout_e24, calc_err(TARGET_ROUT, Rout_ex), calc_err(TARGET_ROUT, Rout_e24)),
    ("Performa AC", "Gain |A_v| (V/V)", TARGET_AV, Av_ex, Av_e24, calc_err(TARGET_AV, Av_ex), calc_err(TARGET_AV, Av_e24)),
    
    # TEST AUTO-CORRECTION DC
    ("Uji Stabilitas", "I_D (+50mV dVg)", "-", "-", ID_50mV * 1000, "-", "-"),
    ("Uji Stabilitas", "I_D (+200mV dVg)", "-", "-", ID_200mV * 1000, "-", "-"),
    ("Uji Stabilitas", "I_D (+500mV dVg)", "-", "-", ID_500mV * 1000, "-", "-")
]

df = pd.DataFrame(data, columns=["Kategori", "Parameter", "Target", "Hasil Eksak", "Hasil E24", "Galat Eksak (%)", "Galat E24 (%)"])

# Formatting khusus agar output bersih dari NaN dan angka desimal rapi
pd.options.display.float_format = '{:.6f}'.format
df.replace("-", np.nan, inplace=True)
df = df.fillna("-")

print("="*105)
print(f"{'RINGKASAN PERHITUNGAN REPRODUCIBLE (EKSAK VS E24)':^105}")
print("="*105)
print(df.to_string(index=False))

In [ ]:
# Inisialisasi LTspice (sesuaikan path jika berbeda)
LTSpice_path = "/Applications/LTspice.app/Contents/MacOS/LTspice"
LTC = LTSpiceBatch(LTSpice_path)
print("LTspice Batch Initialized Successfully.\n")

# Parameter yang akan di-inject
# Kamu tinggal ganti angka-angka di bawah ini saat data 10 orang sudah ada
circuit_params = {
    'my_VTO': V_TH,
    'my_KP': K_N*2,  # Karena LTspice menggunakan KP = 1/2 * μ_n * C_ox * (W/L)
    'my_LAMBDA': LAMBDA,
    'RD_val': RD_e24,
    'RS1_val': RS1_e24,
    'RS2_val': RS2_e24,
    'R1_val': R1_e24,
    'R2_val': R2_e24,
    'Cin_val': 10e-6,
    'Cout_val': 10e-6,
    'Cs_val': 100e-6
}

# Menjalankan simulasi Baseline
LTC.run_netlist("CS_Amp_Baseline.asc", parameters=circuit_params)

print("Simulasi Fase 1 Selesai. Data .raw telah siap dianalisis.")